In [ ]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
from joblib import Parallel, delayed
from pathlib import Path

In [98]:
def gen_truncated_svd(file, output_dir='wframes', figsize=(14.4, 10.8)):
    img = mpimg.imread(file) / 255
    singular_values = min(img.shape[:2])
    k = round(img.mean() * singular_values)
    res = [tuple(np.linalg.svd(img[..., c], full_matrices=False)) for c in range(img.shape[2])]

    plt.subplots_adjust(0, 0, 1, 1)
    plt.figure(figsize=figsize)
    plt.axis("off")
    plt.tight_layout()

    first_sv = 5
    sampled_sv = np.sort(
        np.random.choice(
            np.arange(first_sv + 1, singular_values), 
            size=min(max(k - first_sv, 1), singular_values - first_sv - 1), 
            replace=False
        )
    )
    sampled_sv = np.hstack([np.arange(0, first_sv), sampled_sv])
    plt.imshow(np.clip(np.stack([(u[:, sampled_sv] * s[sampled_sv]) @ v[sampled_sv, :] for u, s, v in res], axis=-1), 0, 1))
    plt.savefig(Path(output_dir) / Path(file).name)
    # plt.show()
    plt.close()
    

In [99]:
plt.style.use('_mpl-gallery-nogrid')
if not Path('wframes').exists():
    Path('wframes').mkdir(parents=True, exist_ok=True)

gen_truncated_svd("frames/output_1700.jpg")
frames = Path('frames').iterdir()
# gen_truncated_svd(next(frames))
next(frames)
Parallel(n_jobs=10)(delayed(gen_truncated_svd)(frame) for frame in frames);

<Figure size 200x200 with 0 Axes>